# Testing Model with Digitized ECGs
This notebook loads your pre-trained `ResNet1D` model and tests it using the `.pt` files created by our ECGtizer pipeline.

**Bug Fix:** The previous `.pt` files failed because they were saved as a dictionary `{'signal': tensor}` and had the shape `(1000, 12)`. Your model requires just the tensor directly with the shape `(12, 1000)`. Both the files and the generating pipeline have been patched to output exactly what the model expects.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import os
import matplotlib.pyplot as plt

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cpu


In [2]:
# ============================================================
# Model Architecture (Copied from learning.ipynb)
# ============================================================
DROPOUT_P = 0.6

class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super(ResidualBlock, self).__init__()
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size=5, stride=stride, padding=2, bias=False)
        self.bn1 = nn.BatchNorm1d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size=5, stride=1, padding=2, bias=False)
        self.bn2 = nn.BatchNorm1d(out_channels)
        self.downsample = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.downsample = nn.Sequential(
                nn.Conv1d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm1d(out_channels)
            )
    def forward(self, x):
        identity = self.downsample(x)
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += identity
        return self.relu(out)

class ResNet1D(nn.Module):
    def __init__(self, n_leads=12, n_classes=4): # n_classes=4 as in learning.ipynb
        super(ResNet1D, self).__init__()
        self.in_channels = 64
        self.conv1 = nn.Conv1d(n_leads, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm1d(64)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool1d(kernel_size=3, stride=2, padding=1)
        self.layer1 = self._make_layer(64, blocks=2, stride=1)
        self.layer2 = self._make_layer(128, blocks=2, stride=2)
        self.layer3 = self._make_layer(256, blocks=2, stride=2)
        self.avgpool = nn.AdaptiveAvgPool1d(1)
        self.dropout = nn.Dropout(DROPOUT_P)
        self.fc = nn.Linear(256, n_classes)

    def _make_layer(self, out_channels, blocks, stride):
        layers = []
        layers.append(ResidualBlock(self.in_channels, out_channels, stride))
        self.in_channels = out_channels
        for _ in range(1, blocks):
            layers.append(ResidualBlock(self.in_channels, out_channels, stride=1))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.maxpool(self.relu(self.bn1(self.conv1(x))))
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.dropout(x)
        return self.fc(x)

In [6]:
# ============================================================
# Load Pre-Trained Model
# ============================================================
MODEL_PATH = r'model/ecg_arrhythmia_model_v3.pt'
OUTFEAT = 4

model = ResNet1D(n_leads=12, n_classes=OUTFEAT).to(device)

if os.path.exists(MODEL_PATH):
    model.load_state_dict(torch.load(MODEL_PATH, map_location=device, weights_only=True))
    model.eval()
    print(f"Successfully loaded model weights from: {MODEL_PATH}")
else:
    print(f"ERROR: Model file not found at {MODEL_PATH}")

Successfully loaded model weights from: model/ecg_arrhythmia_model_v3.pt


In [7]:
# ============================================================
# Predict Function
# ============================================================
def predict_ecg(tensor_path, model):
    print("-"*60)
    print(f"Testing: {os.path.basename(tensor_path)}")
    
    if not os.path.exists(tensor_path):
        print(f"File not found: {tensor_path}")
        return
        
    # 1. Load the tensor
    x = torch.load(tensor_path, map_location=device, weights_only=True)
    print(f"  Loaded shape: {x.shape}")
    
    # 2. Check shape & Add batch dimension
    # Model expects (Batch, Leads, TimeSteps) = (1, 12, 1000)
    if x.shape == (12, 1000):
        x_input = x.unsqueeze(0)  # Add batch dimension
    else:
        print(f"  ERROR: Unexpected shape {x.shape}. Expected (12, 1000).")
        return
        
    x_input = x_input.to(device)
    
    # 3. Run Inference
    with torch.no_grad():
        logits = model(x_input)
        probs = F.softmax(logits, dim=1).squeeze().cpu().numpy()
        pred_cls = probs.argmax()
        
    print(f"  --> PREDICTED CLASS: {pred_cls}")
    for i, p in enumerate(probs):
        bar = '█' * int(p * 40)
        print(f"    Class {i}: {p:.4f}  {bar}")

In [8]:
# ============================================================
# Run Tests
# ============================================================
OUTPUT_DIR = r'ecgtizer_output'

# 1. Test Original ECG (Ground Truth PTB-XL)
orig_path = os.path.join(OUTPUT_DIR, 'original_ecg.pt')
predict_ecg(orig_path, model)

# 2. Test Digitized ECG (Generated by ECGtizer)
digi_path = os.path.join(OUTPUT_DIR, 'digitized_ecg.pt')
predict_ecg(digi_path, model)

print("-"*60)
print("If both files predict the exact same class, the ECGtizer extraction pipeline is a success!")

------------------------------------------------------------
Testing: original_ecg.pt
  Loaded shape: torch.Size([12, 1000])
  --> PREDICTED CLASS: 3
    Class 0: 0.0023  
    Class 1: 0.0010  
    Class 2: 0.0007  
    Class 3: 0.9961  ███████████████████████████████████████
------------------------------------------------------------
Testing: digitized_ecg.pt
  Loaded shape: torch.Size([12, 1000])
  --> PREDICTED CLASS: 3
    Class 0: 0.0001  
    Class 1: 0.0024  
    Class 2: 0.0001  
    Class 3: 0.9974  ███████████████████████████████████████
------------------------------------------------------------
If both files predict the exact same class, the ECGtizer extraction pipeline is a success!
